# GraphRAG: Knowledge Graph-Enhanced Catalog Search

This notebook builds on the standard RAG approach by constructing a **knowledge graph** from the
hardware catalogs. Instead of relying solely on vector similarity, GraphRAG:

1. **Extracts entities and relationships** from document chunks using an LLM
2. **Builds a knowledge graph** with products, specifications, and their connections
3. **Detects communities** of related entities for hierarchical summarization
4. **Retrieves via graph traversal + vector search** for richer context
5. **Compares** GraphRAG vs standard RAG on different query types

## When GraphRAG Shines

| Query Type | Standard RAG | GraphRAG |
|-----------|-------------|----------|
| Specific fact lookup | Great | Good |
| Multi-hop reasoning | Weak | Strong |
| Global/thematic questions | Weak | Strong |
| Cross-document synthesis | Weak | Strong |

## Prerequisites

```bash
pip install -r requirements.txt
ollama pull nomic-embed-text
ollama pull mistral:7b
```

In [ ]:
%pip install -q pymupdf chromadb anthropic openai python-dotenv httpx networkx pyvis


## 1. Setup & Configuration

In [ ]:
# ── Model configuration ───────────────────────────────────────────────
# Each task has its own provider ("anthropic" or "ollama") and model.

EXTRACTION_PROVIDER = "anthropic"
EXTRACTION_MODEL    = "claude-sonnet-4-20250514"

SUMMARY_PROVIDER    = "anthropic"
SUMMARY_MODEL       = "claude-sonnet-4-20250514"

ANSWER_PROVIDER     = "anthropic"
ANSWER_MODEL        = "claude-sonnet-4-20250514"

EMBED_PROVIDER      = "ollama"           # "ollama" or "anthropic"
EMBED_MODEL         = "nomic-embed-text" # ollama: nomic-embed-text | anthropic: voyage-3

# ── Pipeline settings ─────────────────────────────────────────────────

MAX_PAGES_PER_PDF   = None      # None = all pages
FORCE_RE_EXTRACT    = False     # True = ignore cached extractions
CHUNK_SIZE          = 800
CHUNK_OVERLAP       = 100

# ── Paths & connection ────────────────────────────────────────────────

CATALOG_DIR         = "../../catalogs"
CHROMA_PERSIST_DIR  = "../chroma_db_graph"
GRAPH_PERSIST_PATH  = "../knowledge_graph.json"
OLLAMA_BASE_URL     = "http://localhost:11434"

In [ ]:
import os
import json
import re
import time
from pathlib import Path
from difflib import SequenceMatcher
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

# Convert path strings to Path objects
CATALOG_DIR = Path(CATALOG_DIR)
CHROMA_PERSIST_DIR = Path(CHROMA_PERSIST_DIR)
GRAPH_PERSIST_PATH = Path(GRAPH_PERSIST_PATH)

print(f"Anthropic API key: {'loaded' if os.getenv('ANTHROPIC_API_KEY') else 'MISSING'}")
print(f"\nExtraction:          {EXTRACTION_PROVIDER} / {EXTRACTION_MODEL}")
print(f"Community summaries: {SUMMARY_PROVIDER} / {SUMMARY_MODEL}")
print(f"Answer generation:   {ANSWER_PROVIDER} / {ANSWER_MODEL}")
print(f"Embeddings:          {EMBED_PROVIDER} / {EMBED_MODEL}")
print(f"\nPages per PDF: {MAX_PAGES_PER_PDF or 'All'}  |  Force re-extract: {FORCE_RE_EXTRACT}")

## 2. Extract and Chunk PDFs

Same extraction as the standard RAG notebook, but with smaller chunks optimized for entity extraction.

In [ ]:
import fitz  # PyMuPDF


def extract_pdf_pages(pdf_path: Path, max_pages: int | None = None) -> list[dict]:
    """Extract text from each page of a PDF."""
    documents = []
    doc = fitz.open(pdf_path)
    num_pages = min(len(doc), max_pages) if max_pages else len(doc)
    for page_num in range(num_pages):
        page = doc[page_num]
        text = page.get_text().strip()
        if text and len(text) > 50:  # Skip near-blank pages
            documents.append({
                "text": text,
                "metadata": {
                    "source": pdf_path.name,
                    "page": page_num + 1,
                },
            })
    doc.close()
    return documents


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks."""
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            search_start = start + int(chunk_size * 0.8)
            last_break = max(text.rfind(".", search_start, end), text.rfind("\n", search_start, end))
            if last_break > search_start:
                end = last_break + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap
    return chunks


# Extract and chunk
all_chunks = []
for pdf_path in sorted(CATALOG_DIR.glob("*.pdf")):
    pages = extract_pdf_pages(pdf_path, max_pages=MAX_PAGES_PER_PDF)
    for page_doc in pages:
        for i, chunk in enumerate(chunk_text(page_doc["text"])):
            all_chunks.append({
                "id": f"chunk_{len(all_chunks)}",
                "text": chunk,
                "metadata": {**page_doc["metadata"], "chunk_index": i},
            })
    print(f"  {pdf_path.name}: {len(pages)} pages")

print(f"\nTotal chunks: {len(all_chunks)}")

## 3. Entity & Relationship Extraction with LLM

This is the core of GraphRAG. We send each chunk to an LLM and ask it to extract:
- **Entities**: products, manufacturers, specifications, materials, features
- **Relationships**: triples of (subject, predicate, object)

We use Ollama locally so this step is free (but takes time).

In [ ]:
import anthropic
from openai import OpenAI

# Anthropic client
anthropic_client = anthropic.Anthropic()

# Ollama client
ollama_client = OpenAI(
    base_url=f"{OLLAMA_BASE_URL}/v1",
    api_key="ollama",
)


def llm_chat(prompt: str, provider: str, model: str, temperature: float = 0.0, max_tokens: int = 1024) -> str:
    """Send a prompt to the specified provider/model and return the response text."""
    if provider == "anthropic":
        response = anthropic_client.messages.create(
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text.strip()
    elif provider == "ollama":
        response = ollama_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
        )
        return response.choices[0].message.content.strip()
    else:
        raise ValueError(f"Unknown provider: {provider}. Use 'anthropic' or 'ollama'.")


EXTRACTION_PROMPT = """Extract entities and relationships from this hardware catalog text.

ENTITY TYPES (use ONLY these exact values):
  product, manufacturer, feature, specification, material, certification, category

RELATIONSHIP TYPES (use ONLY these exact values):
  has_feature, manufactured_by, has_specification, compatible_with, part_of, variant_of

Return ONLY a valid JSON object with no markdown formatting, no code blocks, no extra text.
Use "relation" as the key for relationship type (not "type").

Example:
{"entities": [{"name": "Tiomos", "type": "product"}], "relationships": [{"source": "Tiomos", "relation": "manufactured_by", "target": "Grass"}]}

TEXT:
$TEXT$

JSON:"""


def extract_entities_and_relations(chunk_text: str, retries: int = 2) -> dict:
    """Extract entities and relationships using EXTRACTION_PROVIDER / EXTRACTION_MODEL."""
    for attempt in range(retries + 1):
        try:
            content = llm_chat(
                EXTRACTION_PROMPT.replace("$TEXT$", chunk_text),
                provider=EXTRACTION_PROVIDER,
                model=EXTRACTION_MODEL,
                temperature=0.0,
                max_tokens=1024,
            )

            # Strip markdown code blocks if present
            content = re.sub(r"^```json\s*", "", content)
            content = re.sub(r"\s*```$", "", content)

            json_match = re.search(r"\{[\s\S]*\}", content)
            if json_match:
                parsed = json.loads(json_match.group())
                entities = parsed.get("entities", [])
                relationships = parsed.get("relationships", [])
                for rel in relationships:
                    if "type" in rel and "relation" not in rel:
                        rel["relation"] = rel.pop("type")
                return {"entities": entities, "relationships": relationships}
            else:
                if attempt == retries:
                    print(f"No JSON in response: {content[:200]}")
        except anthropic.BadRequestError as e:
            error_msg = str(e)
            if "credit balance" in error_msg.lower():
                raise RuntimeError("Anthropic API credits exhausted. Top up at console.anthropic.com/settings/billing") from e
            if attempt == retries:
                print(f"API error: {e}")
        except anthropic.AuthenticationError as e:
            raise RuntimeError(f"Invalid API key. Check ANTHROPIC_API_KEY in .env: {e}") from e
        except anthropic.APIConnectionError as e:
            if attempt == retries:
                print(f"Connection error (check internet): {e}")
        except json.JSONDecodeError as e:
            if attempt == retries:
                print(f"JSON parse error: {e}")
        except Exception as e:
            if attempt == retries:
                print(f"Unexpected error: {type(e).__name__}: {e}")
    return {"entities": [], "relationships": []}


# Quick test
test_result = extract_entities_and_relations(all_chunks[5]["text"])
if test_result["entities"]:
    print(f"Extraction OK ({EXTRACTION_PROVIDER} / {EXTRACTION_MODEL})")
    print(json.dumps(test_result, indent=2))
else:
    print("Warning: test extraction returned no entities. Check API credits and connection.")

## 4. Run Extraction on All Chunks

This is the most time-consuming step. Each chunk is sent through the LLM for entity/relationship extraction. Progress is printed and results are saved to disk so you don't have to re-run.

In [ ]:
# Check if we already have saved extraction results
extractions_path = Path("../extractions.json")

if extractions_path.exists() and not FORCE_RE_EXTRACT:
    print("Loading saved extractions from disk...")
    with open(extractions_path) as f:
        all_extractions = json.load(f)
    print(f"Loaded {len(all_extractions)} extractions")
else:
    print(f"Extracting entities from {len(all_chunks)} chunks...")
    all_extractions = []
    start_time = time.time()
    errors = 0

    for i, chunk in enumerate(all_chunks):
        result = extract_entities_and_relations(chunk["text"])
        result["chunk_id"] = chunk["id"]
        result["source"] = chunk["metadata"]["source"]
        result["page"] = chunk["metadata"]["page"]
        all_extractions.append(result)

        # Abort early if too many consecutive failures
        if not result["entities"] and not result["relationships"]:
            errors += 1
            if errors >= 10 and i < 20:
                print(f"Too many extraction failures ({errors}/{i+1}). Check API credits/connection.")
                break
        else:
            errors = 0

        total_e = sum(len(e.get("entities", [])) for e in all_extractions)
        total_r = sum(len(e.get("relationships", [])) for e in all_extractions)
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        remaining = (len(all_chunks) - i - 1) / rate if rate > 0 else 0
        print(f"  [{i+1}/{len(all_chunks)}] Entities: {total_e}, Relations: {total_r} (~{remaining:.0f}s remaining)")

        if (i + 1) % 50 == 0:
            with open(extractions_path, "w") as f:
                json.dump(all_extractions, f)

    # Only save if we actually got results
    total_e = sum(len(e.get("entities", [])) for e in all_extractions)
    if total_e > 0:
        with open(extractions_path, "w") as f:
            json.dump(all_extractions, f)
        print(f"Saved {len(all_extractions)} extractions")
    else:
        print("No entities extracted - not saving (check API credits)")

print()
print(f"Chunks: {len(all_extractions)}")
total_e = sum(len(e.get("entities", [])) for e in all_extractions)
total_r = sum(len(e.get("relationships", [])) for e in all_extractions)
print(f"Raw entities: {total_e}, Raw relationships: {total_r}")

## 4a. Post-Process Extractions

Normalize entity types and relationship types to a small fixed set, deduplicate entities via fuzzy matching, and filter noise. This dramatically improves graph quality.

In [ ]:
# === Normalize entity types ===
ENTITY_TYPE_MAP = {
    "product": "product", "product_model": "product", "product_variant": "product",
    "product_code": "product", "product_id": "product", "product_series": "product",
    "product_line": "product", "product_component": "product", "component": "product",
    "accessory": "product", "accessory_item": "product", "part": "product",
    "variant": "product", "hinge_style": "product", "tool": "product",
    "item_number": "product",
    "manufacturer": "manufacturer", "company": "manufacturer", "brand": "manufacturer",
    "supplier": "manufacturer",
    "feature": "feature", "product_feature": "feature", "function": "feature",
    "technology": "feature", "action": "feature", "property": "feature",
    "attribute": "feature", "description": "feature",
    "specification": "specification", "product_specification": "specification",
    "dimension": "specification", "weight": "specification", "size": "specification",
    "value": "specification", "quantity": "specification", "unit_of_measurement": "specification",
    "unit_of_measure": "specification", "pattern": "specification",
    "drill_pattern": "specification", "drilling_pattern": "specification",
    "fixing_method": "specification", "number": "specification",
    "material": "material",
    "certification": "certification", "standard": "certification",
    "category": "category", "section": "category", "application": "category",
    "product_application": "category", "usage": "category", "cabinetry": "category",
    "context": "category", "location": "category", "document": "category",
}

# === Normalize relationship types ===
RELATION_TYPE_MAP = {
    "has_feature": "has_feature", "includes_feature": "has_feature",
    "includes": "has_feature", "has_close_type": "has_feature",
    "includes_technology": "has_feature", "has_opening": "has_feature",
    "has_position": "has_feature", "provides": "has_feature",
    "enables": "has_feature", "controls": "has_feature",
    "integrated_with": "has_feature", "has_power_factor_range": "has_feature",
    "offers": "has_feature",
    "has_specification": "has_specification", "has_dimension": "has_specification",
    "has_weight": "has_specification", "has_pattern": "has_specification",
    "has_fixing": "has_specification", "has_unit_of_measure": "has_specification",
    "has_description": "has_specification", "has_cabinet_height_range": "has_specification",
    "is_unit_of": "has_specification", "has": "has_specification",
    "has_number": "has_specification",
    "part_of": "part_of", "is_part_of": "part_of", "contains": "part_of",
    "consists_of": "part_of", "belongs_to": "part_of",
    "compatible_with": "compatible_with", "suitable_for": "compatible_with",
    "suited_for": "compatible_with", "applies_to": "compatible_with",
    "applicable_to": "compatible_with", "used_for": "compatible_with",
    "recommended_for": "compatible_with", "recommends": "compatible_with",
    "attaches_to": "compatible_with", "found_on": "compatible_with",
    "for": "compatible_with",
    "manufactured_by": "manufactured_by",
    "variant_of": "variant_of", "is_variant_of": "variant_of",
    "made_of": "has_specification",
}

# Apply normalization
unmapped_types = set()
unmapped_rels = set()

for extraction in all_extractions:
    for entity in extraction.get("entities", []):
        raw_type = str(entity.get("type", "unknown")).lower().strip()
        mapped = ENTITY_TYPE_MAP.get(raw_type)
        if mapped:
            entity["type"] = mapped
        else:
            if raw_type is not None: unmapped_types.add(raw_type)
            entity["type"] = "feature"

    for rel in extraction.get("relationships", []):
        raw_rel = rel.get("relation", "related_to")
        if raw_rel:
            raw_rel = str(raw_rel).lower().strip()
        mapped = RELATION_TYPE_MAP.get(raw_rel)
        if mapped:
            rel["relation"] = mapped
        else:
            if raw_rel is not None: unmapped_rels.add(raw_rel)
            rel["relation"] = "has_feature"

    # Filter out relationships with None/empty/non-string source or target
    extraction["relationships"] = [
        r for r in extraction.get("relationships", [])
        if r.get("source") and r.get("target")
        and isinstance(r.get("source"), str) and isinstance(r.get("target"), str)
    ]

print("Entity type normalization complete")
if unmapped_types:
    print(f"  Unmapped types (defaulted to 'feature'): {sorted(unmapped_types)}")

print("\nRelationship type normalization complete")
if unmapped_rels:
    print(f"  Unmapped relations (defaulted to 'has_feature'): {sorted(unmapped_rels)}")

# Show new distributions
from collections import Counter
type_counts = Counter()
rel_counts = Counter()
for ex in all_extractions:
    for e in ex.get("entities", []):
        type_counts[e["type"]] += 1
    for r in ex.get("relationships", []):
        rel_counts[r["relation"]] += 1

print(f"\nEntity types ({len(type_counts)}): {dict(type_counts.most_common())}")
print(f"Relationship types ({len(rel_counts)}): {dict(rel_counts.most_common())}")

In [ ]:
# === Entity deduplication via fuzzy matching ===

def build_merge_map(extractions):
    name_counts = {}
    for ex in extractions:
        for e in ex.get("entities", []):
            name = str(e.get("name", "")).strip().title()
            if name:
                name_counts[name] = name_counts.get(name, 0) + 1
        for r in ex.get("relationships", []):
            for key in ["source", "target"]:
                val = r.get(key)
                if val and isinstance(val, str):
                    name = val.strip().title()
                    name_counts[name] = name_counts.get(name, 0) + 1

    names = sorted(name_counts.keys())
    merge_map = {}

    for i, name_a in enumerate(names):
        if name_a in merge_map:
            continue
        for name_b in names[i+1:]:
            if name_b in merge_map:
                continue

            a_norm = name_a.lower().replace("-", " ").replace("_", " ").strip()
            b_norm = name_b.lower().replace("-", " ").replace("_", " ").strip()

            if a_norm == b_norm:
                canonical = name_a if name_counts[name_a] >= name_counts[name_b] else name_b
                variant = name_b if canonical == name_a else name_a
                merge_map[variant] = canonical
                continue

            if len(a_norm) < 30 and len(b_norm) < 30 and len(a_norm) > 3 and len(b_norm) > 3:
                ratio = SequenceMatcher(None, a_norm, b_norm).ratio()
                if ratio >= 0.92:
                    canonical = name_a if name_counts[name_a] >= name_counts[name_b] else name_b
                    variant = name_b if canonical == name_a else name_a
                    merge_map[variant] = canonical

    return merge_map


merge_map = build_merge_map(all_extractions)
print(f"Merging {len(merge_map)} entity variants:")
for v, c in sorted(merge_map.items()):
    print(f"  '{v}' -> '{c}'")

for ex in all_extractions:
    for e in ex.get("entities", []):
        name = str(e.get("name", "")).strip().title()
        e["name"] = merge_map.get(name, name)
    for r in ex.get("relationships", []):
        for key in ["source", "target"]:
            val = r.get(key)
            if val and isinstance(val, str):
                name = val.strip().title()
                r[key] = merge_map.get(name, name)

print(f"\nDeduplication applied to all extractions")

In [ ]:
# === Filter low-value / noise entities ===
STOP_ENTITIES = {
    "N/A", "None", "Unknown", "Other", "Yes", "No", "See", "Page",
    "Table", "Figure", "Note", "Section", "Mm", "Kg", "Lbs", "Item #",
    "Contents", "Index", "Catalog", "Overview",
}

before_entities = sum(len(ex.get("entities", [])) for ex in all_extractions)
before_rels = sum(len(ex.get("relationships", [])) for ex in all_extractions)

for ex in all_extractions:
    ex["entities"] = [
        e for e in ex.get("entities", [])
        if isinstance(e.get("name"), str)
        and len(e["name"].strip()) >= 3
        and e["name"].strip().title() not in STOP_ENTITIES
        and not e["name"].strip().replace(".", "").replace(",", "").replace("/", "").replace("-", "").replace(" ", "").isdigit()
    ]
    ex["relationships"] = [
        r for r in ex.get("relationships", [])
        if isinstance(r.get("source"), str) and len(r["source"].strip()) >= 3
        and isinstance(r.get("target"), str) and len(r["target"].strip()) >= 3
    ]

after_entities = sum(len(ex.get("entities", [])) for ex in all_extractions)
after_rels = sum(len(ex.get("relationships", [])) for ex in all_extractions)

print(f"Filtered entities: {before_entities} -> {after_entities} (removed {before_entities - after_entities})")
print(f"Filtered relationships: {before_rels} -> {after_rels} (removed {before_rels - after_rels})")

## 5. Build the Knowledge Graph

We use NetworkX to construct a graph from the extracted entities and relationships. Entity names are normalized to merge duplicates.

In [ ]:
import networkx as nx


def normalize_name(name: str) -> str:
    """Normalize entity names for consistent matching."""
    if not isinstance(name, str):
        name = str(name) if name is not None else ""
    return name.strip().title()


# Build the graph
G = nx.Graph()

for extraction in all_extractions:
    source_file = extraction["source"]
    page = extraction["page"]
    chunk_id = extraction["chunk_id"]

    # Add entity nodes
    for entity in extraction["entities"]:
        name = normalize_name(entity.get("name", ""))
        etype = entity.get("type", "unknown")
        if not name or len(name) < 2:
            continue

        if G.has_node(name):
            G.nodes[name]["mentions"] = G.nodes[name].get("mentions", 1) + 1
            G.nodes[name]["sources"].add(source_file)
            G.nodes[name]["chunk_ids"].add(chunk_id)
            # Type majority voting
            G.nodes[name]["type_votes"][etype] = G.nodes[name]["type_votes"].get(etype, 0) + 1
            G.nodes[name]["type"] = max(G.nodes[name]["type_votes"], key=G.nodes[name]["type_votes"].get)
        else:
            G.add_node(
                name,
                type=etype,
                mentions=1,
                sources={source_file},
                chunk_ids={chunk_id},
                type_votes={etype: 1},
            )

    # Add relationship edges
    for rel in extraction["relationships"]:
        src = normalize_name(rel.get("source", ""))
        tgt = normalize_name(rel.get("target", ""))
        relation = rel.get("relation", "has_feature")
        if not src or not tgt or len(src) < 2 or len(tgt) < 2:
            continue

        for node_name in [src, tgt]:
            if not G.has_node(node_name):
                # Infer type: if it appears as an entity elsewhere, use that type
                inferred_type = "product"  # default for relationship endpoints
                G.add_node(node_name, type=inferred_type, mentions=1, sources={source_file},
                           chunk_ids={chunk_id}, type_votes={inferred_type: 1})

        if G.has_edge(src, tgt):
            G.edges[src, tgt]["weight"] = G.edges[src, tgt].get("weight", 1) + 1
            G.edges[src, tgt]["relations"].add(relation)
        else:
            G.add_edge(src, tgt, weight=1, relations={relation})

print(f"Knowledge Graph:")
print(f"  Nodes (entities): {G.number_of_nodes()}")
print(f"  Edges (relationships): {G.number_of_edges()}")
print(f"  Connected components: {nx.number_connected_components(G)}")
print(f"  Average degree: {sum(d for _, d in G.degree()) / max(G.number_of_nodes(), 1):.1f}")
print(f"  Density: {nx.density(G):.4f}")

# Entity type distribution
from collections import Counter
type_dist = Counter(data.get("type", "?") for _, data in G.nodes(data=True))
print(f"\nEntity types: {dict(type_dist.most_common())}")

# Relationship type distribution
rel_dist = Counter()
for _, _, data in G.edges(data=True):
    for r in data.get("relations", set()):
        rel_dist[r] += 1
print(f"Relationship types: {dict(rel_dist.most_common())}")

# Top entities by mention count
top_entities = sorted(G.nodes(data=True), key=lambda x: x[1].get("mentions", 0), reverse=True)[:15]
print(f"\nTop 15 entities:")
for name, data in top_entities:
    print(f"  {name} ({data.get('type', '?')}): {data.get('mentions', 0)} mentions, {len(data.get('sources', set()))} sources")

## 6. Community Detection

We detect communities (clusters of densely connected entities) using the Louvain algorithm. Each community represents a thematic grouping — e.g., a product line with its features and specs.

In [ ]:
# Detect communities using Louvain method
communities = nx.community.louvain_communities(G, seed=42)

# Assign community IDs to nodes
for community_id, community_nodes in enumerate(communities):
    for node in community_nodes:
        G.nodes[node]["community"] = community_id

print(f"Detected {len(communities)} communities\n")

# Show the largest communities
sorted_communities = sorted(enumerate(communities), key=lambda x: len(x[1]), reverse=True)

for community_id, members in sorted_communities[:8]:
    # Get top members by mentions
    top_members = sorted(members, key=lambda n: G.nodes[n].get("mentions", 0), reverse=True)[:6]
    member_str = ", ".join(top_members)
    if len(members) > 6:
        member_str += f", ... (+{len(members)-6} more)"
    print(f"Community {community_id} ({len(members)} entities): {member_str}")

## 7. Visualize the Knowledge Graph

Interactive visualization using pyvis. Nodes are colored by community and sized by mention count.

In [ ]:
from pyvis.network import Network

# Build a subgraph of the most-connected entities for a readable visualization
# (full graph can be overwhelming)
MIN_MENTIONS = 3  # Only include entities mentioned at least this many times
notable_nodes = [n for n, d in G.nodes(data=True) if d.get("mentions", 0) >= MIN_MENTIONS]
subgraph = G.subgraph(notable_nodes)

print(f"Visualizing subgraph: {subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges")
print(f"(Entities with >= {MIN_MENTIONS} mentions)\n")

# Color palette for communities
COLORS = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990",
    "#dcbeff", "#9A6324", "#800000", "#aaffc3", "#808000",
]

net = Network(height="700px", width="100%", notebook=True, cdn_resources="in_line")
net.barnes_hut(gravity=-5000, spring_length=200)

for node in subgraph.nodes():
    data = G.nodes[node]
    community_id = data.get("community", 0)
    color = COLORS[community_id % len(COLORS)]
    size = min(10 + data.get("mentions", 1) * 3, 50)
    title = f"{node}\nType: {data.get('type', '?')}\nMentions: {data.get('mentions', 0)}\nCommunity: {community_id}"
    net.add_node(node, label=node, color=color, size=size, title=title)

for src, tgt, data in subgraph.edges(data=True):
    relations = ", ".join(data.get("relations", set()))
    net.add_edge(src, tgt, title=relations, width=min(data.get("weight", 1), 5))

# Write HTML with explicit UTF-8 encoding (fixes Windows cp1252 error)
html_path = Path("../knowledge_graph.html")
net.generate_html()
with open(html_path, "w", encoding="utf-8") as f:
    f.write(net.html)

print(f"Graph saved to {html_path.resolve()}")
print("Open it in a browser for the interactive visualization.")

## 8. Community Summaries

Generate natural language summaries for each community. These summaries enable answering broad, thematic questions that standard RAG struggles with.

In [ ]:
SUMMARY_PROMPT = """You are summarizing a cluster of related entities from hardware product catalogs.

Given the following entities and their relationships, write a concise summary (2-4 sentences)
describing what this group represents, the key products/features, and how they relate.

ENTITIES:
{entities}

RELATIONSHIPS:
{relationships}

SUMMARY:"""


def summarize_community(community_nodes: set, graph: nx.Graph) -> str:
    """Generate a natural language summary for a community."""
    # Collect entity info
    entity_strs = []
    for node in sorted(community_nodes, key=lambda n: graph.nodes[n].get("mentions", 0), reverse=True)[:20]:
        data = graph.nodes[node]
        entity_strs.append(f"- {node} (type: {data.get('type', '?')}, mentions: {data.get('mentions', 0)})")

    # Collect relationships within the community
    rel_strs = []
    for src, tgt, data in graph.edges(data=True):
        if src in community_nodes and tgt in community_nodes:
            relations = ", ".join(data.get("relations", set()))
            rel_strs.append(f"- {src} --[{relations}]--> {tgt}")

    if not entity_strs:
        return "Empty community."

    prompt = SUMMARY_PROMPT.format(
        entities="\n".join(entity_strs[:20]),
        relationships="\n".join(rel_strs[:30]) or "No relationships found.",
    )

    return llm_chat(prompt, provider=SUMMARY_PROVIDER, model=SUMMARY_MODEL, temperature=0.3, max_tokens=512)


# Summarize the top communities (by size)
community_summaries = {}
top_n_communities = min(10, len(sorted_communities))

print(f"Generating summaries for top {top_n_communities} communities ({SUMMARY_PROVIDER} / {SUMMARY_MODEL})...\n")
for community_id, members in sorted_communities[:top_n_communities]:
    if len(members) < 3:  # Skip tiny communities
        continue
    summary = summarize_community(members, G)
    community_summaries[community_id] = {
        "summary": summary,
        "size": len(members),
        "top_entities": sorted(members, key=lambda n: G.nodes[n].get("mentions", 0), reverse=True)[:5],
    }
    print(f"Community {community_id} ({len(members)} entities):")
    print(f"  Key entities: {', '.join(community_summaries[community_id]['top_entities'])}")
    print(f"  Summary: {summary}\n")

## 9. Build ChromaDB Vector Store

We store both the original chunks AND the community summaries in ChromaDB. This gives us vector search over raw text plus graph-derived knowledge.

In [ ]:
import httpx
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings


class OllamaEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model: str = EMBED_MODEL, base_url: str = OLLAMA_BASE_URL):
        self.model = model
        self.base_url = base_url

    def __call__(self, input: Documents) -> Embeddings:
        embeddings = []
        batch_size = 32
        for i in range(0, len(input), batch_size):
            batch = input[i : i + batch_size]
            response = httpx.post(
                f"{self.base_url}/api/embed",
                json={"model": self.model, "input": batch},
                timeout=120.0,
            )
            response.raise_for_status()
            embeddings.extend(response.json()["embeddings"])
        return embeddings


class AnthropicEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model: str = EMBED_MODEL):
        import voyageai
        self.client = voyageai.Client()
        self.model = model

    def __call__(self, input: Documents) -> Embeddings:
        embeddings = []
        batch_size = 32
        for i in range(0, len(input), batch_size):
            batch = input[i : i + batch_size]
            result = self.client.embed(batch, model=self.model)
            embeddings.extend(result.embeddings)
        return embeddings


if EMBED_PROVIDER == "ollama":
    embed_fn = OllamaEmbeddingFunction()
elif EMBED_PROVIDER == "anthropic":
    embed_fn = AnthropicEmbeddingFunction()
else:
    raise ValueError(f"Unknown EMBED_PROVIDER: {EMBED_PROVIDER}. Use 'ollama' or 'anthropic'.")

print(f"Embedding: {EMBED_PROVIDER} / {EMBED_MODEL}")

chroma_client = chromadb.PersistentClient(path=str(CHROMA_PERSIST_DIR.resolve()))

# --- Chunks collection ---
try:
    chroma_client.delete_collection("graph_rag_chunks")
except ValueError:
    pass

chunks_collection = chroma_client.create_collection(
    name="graph_rag_chunks",
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"},
)

print(f"Adding {len(all_chunks)} chunks...")
BATCH_SIZE = 50
for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i : i + BATCH_SIZE]
    chunks_collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[c["metadata"] for c in batch],
    )
    print(f"  {min(i + BATCH_SIZE, len(all_chunks))}/{len(all_chunks)}", end="\r")

# --- Community summaries collection ---
try:
    chroma_client.delete_collection("graph_rag_communities")
except ValueError:
    pass

community_collection = chroma_client.create_collection(
    name="graph_rag_communities",
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"},
)

if community_summaries:
    community_collection.add(
        ids=[f"community_{cid}" for cid in community_summaries],
        documents=[cs["summary"] for cs in community_summaries.values()],
        metadatas=[{"community_id": cid, "size": cs["size"], "top_entities": ", ".join(cs["top_entities"])} for cid, cs in community_summaries.items()],
    )

print(f"\nChunks collection: {chunks_collection.count()} documents")
print(f"Community collection: {community_collection.count()} summaries")

## 10. GraphRAG Retrieval

The GraphRAG retrieval strategy combines three signals:

1. **Vector search** — find relevant text chunks (same as standard RAG)
2. **Graph traversal** — find entities mentioned in top chunks, then pull in their neighbors and connected information
3. **Community summaries** — find relevant high-level summaries for thematic context

In [ ]:
def graph_retrieve(query: str, n_chunks: int = 5, n_communities: int = 2, hop_depth: int = 1) -> dict:
    """GraphRAG retrieval: vector search + graph traversal + community summaries."""

    # 1. Vector search on chunks
    chunk_results = chunks_collection.query(query_texts=[query], n_results=n_chunks)
    retrieved_chunks = []
    for i in range(len(chunk_results["ids"][0])):
        retrieved_chunks.append({
            "id": chunk_results["ids"][0][i],
            "text": chunk_results["documents"][0][i],
            "metadata": chunk_results["metadatas"][0][i],
            "distance": chunk_results["distances"][0][i],
        })

    # 2. Graph traversal: find entities in retrieved chunks, get their neighbors
    retrieved_chunk_ids = set(chunk_results["ids"][0])
    graph_entities = set()
    graph_context_parts = []

    for node, data in G.nodes(data=True):
        node_chunks = data.get("chunk_ids", set())
        if node_chunks & retrieved_chunk_ids:
            graph_entities.add(node)

            # Get neighbors up to hop_depth
            neighbors = set()
            current_level = {node}
            for _ in range(hop_depth):
                next_level = set()
                for n in current_level:
                    next_level.update(G.neighbors(n))
                neighbors.update(next_level)
                current_level = next_level

            graph_entities.update(neighbors)

    # Build graph context: entity descriptions and their relationships
    for entity in sorted(graph_entities, key=lambda n: G.nodes[n].get("mentions", 0), reverse=True)[:15]:
        node_data = G.nodes[entity]
        # Get edges for this entity
        edges = []
        for _, neighbor, edge_data in G.edges(entity, data=True):
            if neighbor in graph_entities:
                rels = ", ".join(edge_data.get("relations", set()))
                edges.append(f"{entity} --[{rels}]--> {neighbor}")

        if edges:
            graph_context_parts.append(
                f"{entity} ({node_data.get('type', '?')}, {node_data.get('mentions', 0)} mentions):\n"
                + "\n".join(f"  {e}" for e in edges[:5])
            )

    # 3. Community summaries
    community_results = community_collection.query(query_texts=[query], n_results=n_communities)
    retrieved_communities = []
    for i in range(len(community_results["ids"][0])):
        retrieved_communities.append({
            "id": community_results["ids"][0][i],
            "summary": community_results["documents"][0][i],
            "metadata": community_results["metadatas"][0][i],
            "distance": community_results["distances"][0][i],
        })

    return {
        "chunks": retrieved_chunks,
        "graph_context": "\n\n".join(graph_context_parts),
        "graph_entities_found": len(graph_entities),
        "communities": retrieved_communities,
    }


# Test it
result = graph_retrieve("What soft-close hinge options are available?")
print(f"Chunks retrieved: {len(result['chunks'])}")
print(f"Graph entities found: {result['graph_entities_found']}")
print(f"Communities matched: {len(result['communities'])}")
print(f"\nGraph context preview:\n{result['graph_context'][:500]}")

## 11. GraphRAG Generation

Build the prompt with all three context sources and generate answers with both Ollama and Claude.

In [ ]:
def build_graph_rag_prompt(query: str, retrieval: dict) -> str:
    """Build a prompt combining chunk context, graph context, and community summaries."""

    # Chunk context (same as standard RAG)
    chunk_parts = []
    for i, chunk in enumerate(retrieval["chunks"], 1):
        src = chunk["metadata"]["source"]
        page = chunk["metadata"]["page"]
        chunk_parts.append(f"[Source {i}: {src}, Page {page}]\n{chunk['text']}")
    chunk_context = "\n\n---\n\n".join(chunk_parts)

    # Graph context
    graph_context = retrieval["graph_context"] or "No graph relationships found."

    # Community summaries
    community_context = "\n".join(
        f"- {c['summary']}" for c in retrieval["communities"]
    ) or "No community summaries available."

    return f"""You are a knowledgeable hardware product specialist. Answer the user's question
using ALL the provided context. You have three sources of information:

1. CATALOG TEXT — direct excerpts from product catalogs
2. KNOWLEDGE GRAPH — extracted entity relationships showing how products, features, and specs connect
3. THEMATIC SUMMARIES — high-level summaries of product clusters

Use all three to give a comprehensive answer. Cite source catalogs and page numbers when possible.
If the context doesn't contain enough information, say so clearly.

=== CATALOG TEXT ===
{chunk_context}

=== KNOWLEDGE GRAPH (entity relationships) ===
{graph_context}

=== THEMATIC SUMMARIES ===
{community_context}

USER QUESTION: {query}

ANSWER:"""


def graph_rag_query(query: str) -> str:
    """Full GraphRAG pipeline using ANSWER_PROVIDER / ANSWER_MODEL."""
    retrieval = graph_retrieve(query)
    prompt = build_graph_rag_prompt(query, retrieval)
    return llm_chat(prompt, provider=ANSWER_PROVIDER, model=ANSWER_MODEL, temperature=0.3, max_tokens=1024)


# Test
query = "What soft-close hinge options are available and how do they compare?"
print(f"Query: {query}")
print(f"Answer: {ANSWER_PROVIDER} / {ANSWER_MODEL}\n")
print("=" * 60)
print(graph_rag_query(query))

## 12. Compare: Standard RAG vs GraphRAG

We test both approaches on queries that highlight GraphRAG's strengths: broad, thematic, and cross-document questions.

In [ ]:
def standard_rag_query(query: str, n_results: int = 5) -> str:
    """Standard RAG (vector-only) for comparison. Uses ANSWER_PROVIDER / ANSWER_MODEL."""
    results = chunks_collection.query(query_texts=[query], n_results=n_results)
    context_parts = []
    for i in range(len(results["ids"][0])):
        src = results["metadatas"][0][i]["source"]
        page = results["metadatas"][0][i]["page"]
        context_parts.append(f"[Source {i+1}: {src}, Page {page}]\n{results['documents'][0][i]}")

    context = "\n\n---\n\n".join(context_parts)
    prompt = f"""You are a knowledgeable hardware product specialist. Answer the user's question
based ONLY on the provided catalog context. If the context doesn't contain enough
information to fully answer the question, say so clearly.

When referencing specific products or specifications, cite the source catalog and page number.

CONTEXT FROM PRODUCT CATALOGS:
{context}

USER QUESTION: {query}

ANSWER:"""

    return llm_chat(prompt, provider=ANSWER_PROVIDER, model=ANSWER_MODEL, temperature=0.3, max_tokens=1024)


# Comparison queries — designed to test different strengths
comparison_queries = [
    # Broad/thematic — GraphRAG advantage
    "What are the main product categories across all catalogs and how do they relate?",
    # Cross-document — GraphRAG advantage
    "Compare the hinge systems from Grass and Wurth Baer.",
    # Multi-hop — GraphRAG advantage
    "What mounting plates are compatible with soft-close hinges?",
    # Specific fact — Standard RAG is fine
    "What is the maximum opening angle for Tiomos hinges?",
]

print(f"Answer provider: {ANSWER_PROVIDER} / {ANSWER_MODEL}\n")

for query in comparison_queries:
    print("\n" + "#" * 70)
    print(f"QUERY: {query}")
    print("#" * 70)

    print(f"\n--- Standard RAG ---")
    print(standard_rag_query(query))

    print(f"\n--- GraphRAG ---")
    print(graph_rag_query(query))
    print()

## 13. Interactive GraphRAG Query

In [ ]:
# Change this query
my_query = "What are all the different hinge types and their key specifications?"

print(f"Query: {my_query}")
print(f"Answer provider: {ANSWER_PROVIDER}\n")

# Retrieve with full GraphRAG
retrieval = graph_retrieve(my_query)

print(f"Retrieved {len(retrieval['chunks'])} chunks")
print(f"Found {retrieval['graph_entities_found']} related entities in knowledge graph")
print(f"Matched {len(retrieval['communities'])} community summaries")

print("\nSources:")
for c in retrieval["chunks"]:
    print(f"  - {c['metadata']['source']}, Page {c['metadata']['page']} (distance: {c['distance']:.4f})")
print()

print(graph_rag_query(my_query))

## 14. Explore the Knowledge Graph

Utilities for querying the graph directly.

In [ ]:
def explore_entity(name: str, depth: int = 1):
    """Show an entity's connections in the knowledge graph."""
    name = normalize_name(name)
    if name not in G:
        # Fuzzy match
        matches = [n for n in G.nodes() if name.lower() in n.lower()]
        if matches:
            print(f"'{name}' not found exactly. Did you mean: {', '.join(matches[:5])}?")
            name = matches[0]
            print(f"Showing results for '{name}':\n")
        else:
            print(f"Entity '{name}' not found in graph.")
            return

    data = G.nodes[name]
    print(f"Entity: {name}")
    print(f"  Type: {data.get('type', '?')}")
    print(f"  Mentions: {data.get('mentions', 0)}")
    print(f"  Sources: {', '.join(data.get('sources', set()))}")
    print(f"  Community: {data.get('community', '?')}")
    print(f"  Connections:")

    for neighbor in sorted(G.neighbors(name), key=lambda n: G.nodes[n].get("mentions", 0), reverse=True):
        edge_data = G.edges[name, neighbor]
        rels = ", ".join(edge_data.get("relations", set()))
        n_data = G.nodes[neighbor]
        print(f"    --[{rels}]--> {neighbor} ({n_data.get('type', '?')}, {n_data.get('mentions', 0)} mentions)")


# Try it
explore_entity("Tiomos")

In [ ]:
# Find shortest path between two entities
def find_path(entity_a: str, entity_b: str):
    """Find the shortest path between two entities in the knowledge graph."""
    a = normalize_name(entity_a)
    b = normalize_name(entity_b)

    # Fuzzy match
    for name in [a, b]:
        if name not in G:
            matches = [n for n in G.nodes() if name.lower() in n.lower()]
            if matches:
                if name == a:
                    a = matches[0]
                else:
                    b = matches[0]

    try:
        path = nx.shortest_path(G, a, b)
        print(f"Path from '{a}' to '{b}' ({len(path)-1} hops):\n")
        for i in range(len(path) - 1):
            edge = G.edges[path[i], path[i+1]]
            rels = ", ".join(edge.get("relations", set()))
            print(f"  {path[i]} --[{rels}]--> {path[i+1]}")
    except (nx.NetworkXNoPath, nx.NodeNotFound) as e:
        print(f"No path found: {e}")


find_path("Tiomos", "Soft Close")

In [ ]:
# Graph statistics
print("Knowledge Graph Statistics")
print("=" * 40)
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Connected components: {nx.number_connected_components(G)}")
print(f"Communities: {len(communities)}")
if G.number_of_nodes() > 0:
    print(f"Average degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.1f}")
    print(f"Density: {nx.density(G):.4f}")

# Entity type distribution
type_counts = {}
for _, data in G.nodes(data=True):
    t = data.get("type", "unknown")
    type_counts[t] = type_counts.get(t, 0) + 1

print(f"\nEntity types:")
for t, count in sorted(type_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {t}: {count}")

# Relationship type distribution
rel_counts = {}
for _, _, data in G.edges(data=True):
    for r in data.get("relations", set()):
        rel_counts[r] = rel_counts.get(r, 0) + 1

print(f"\nRelationship types:")
for r, count in sorted(rel_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {r}: {count}")